# Lab: Bring your own script with Amazon SageMaker

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.0.0</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.7.1</strong></li>
</ul>
</div>

## TensorFlow script mode training and serving

Script mode is a training script format for TensorFlow that lets you execute any TensorFlow training script in SageMaker with minimal modification. The SageMaker Python SDK handles transferring your script to a SageMaker training instance. On the training instance, SageMaker's native TensorFlow support sets up training-related environment variables and executes your training script.

In this example, we use a Python script to train a classification model on the [MNIST dataset](http://yann.lecun.com/exdb/mnist/). We will show how to:

- Use `ModelTrainer` to run distributed training with TensorFlow
- Use `ModelBuilder` to deploy the model as a real-time endpoint
- Invoke the endpoint for predictions

The TensorFlow Serving container is used for inference, which natively supports TensorFlow SavedModel format without requiring custom inference code.

### Install packages
Please choose `Python 3 (ipykernel)` kernel to proceed.

We will first install the prerequisite packages. They should be already installed if you run the [Setup.ipynb](../Setup.ipynb) notebook. If you experience problems, uncomment the following three cells to re-install the right libraires and the restart the kernel and then continue

In [ ]:
## Uncomment and execute if having errors import any of the sagemaker libraries
# %pip install --no-cache-dir sagemaker-core==2.7.1 \
#     sagemaker-train==1.7.1 \
#     sagemaker-serve==1.7.1 --extra-index-url https://download.pytorch.org/whl/cpu \
#     sagemaker-mlops==1.7.1 \
#     sagemaker==3.7.1

In [ ]:
# # Restart kernel to get the packages
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

# Set up the environment

We start by creating a `Session` object which manages interactions with SageMaker APIs. The session provides the default S3 bucket for storing data and model artifacts.

We also retrieve the IAM execution role using `get_execution_role()`, which grants SageMaker permission to access AWS resources.

In [ ]:
# cell 01
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()

role = get_execution_role()
region = sagemaker_session.boto_session.region_name

# Training Data
The MNIST dataset has been loaded to the public S3 buckets `sagemaker-sample-data-<REGION>` under the prefix `tensorflow/mnist`. There are four .npy file under this prefix:

- train_data.npy
- eval_data.npy
- train_labels.npy
- eval_labels.npy

In [ ]:
# cell 02
training_data_uri = 's3://sagemaker-sample-data-{}/tensorflow/mnist'.format(region)

# Construct a script for distributed training

The training script runs inside a SageMaker TensorFlow container. It accesses the training environment through environment variables:

| Variable | Description |
|----------|-------------|
| `SM_MODEL_DIR` | Directory to save model artifacts (always `/opt/ml/model`) |
| `SM_CHANNEL_TRAINING` | Directory containing the training data |
| `SM_HOSTS` | List of all hosts (for distributed training) |
| `SM_CURRENT_HOST` | Current host name |

The script saves the trained model in TensorFlow SavedModel format to `SM_MODEL_DIR`. SageMaker automatically uploads it to S3 after training.

**Important:** For TensorFlow Serving, the model must be saved with a version number subdirectory (e.g., `000000001`).

In [ ]:
!mkdir -p mnist-tensorflow

In [ ]:
%%writefile mnist-tensorflow/train.py
# Copyright 2019 Amazon.com, Inc. or its affiliates. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License"). You
# may not use this file except in compliance with the License. A copy of
# the License is located at
#
#     http://aws.amazon.com/apache2.0/
#
# or in the "license" file accompanying this file. This file is
# distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF
# ANY KIND, either express or implied. See the License for the specific
# language governing permissions and limitations under the License.

import tensorflow as tf
import argparse
import os
import numpy as np
import json


def model(x_train, y_train, x_test, y_test):
    """Generate a simple model"""
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(10, activation=tf.nn.softmax)
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    model.fit(x_train, y_train)
    model.evaluate(x_test, y_test)

    return model


def _load_training_data(base_dir):
    """Load MNIST training data"""
    x_train = np.load(os.path.join(base_dir, 'train_data.npy'))
    y_train = np.load(os.path.join(base_dir, 'train_labels.npy'))
    return x_train, y_train


def _load_testing_data(base_dir):
    """Load MNIST testing data"""
    x_test = np.load(os.path.join(base_dir, 'eval_data.npy'))
    y_test = np.load(os.path.join(base_dir, 'eval_labels.npy'))
    return x_test, y_test


def _parse_args():
    parser = argparse.ArgumentParser()

    # Data, model, and output directories
    # model_dir is always passed in from SageMaker. By default this is a S3 path under the default bucket.
    parser.add_argument('--model_dir', type=str)
    parser.add_argument('--sm-model-dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAINING'))
    parser.add_argument('--hosts', type=list, default=json.loads(os.environ.get('SM_HOSTS')))
    parser.add_argument('--current-host', type=str, default=os.environ.get('SM_CURRENT_HOST'))

    return parser.parse_known_args()


if __name__ == "__main__":
    args, unknown = _parse_args()

    train_data, train_labels = _load_training_data(args.train)
    eval_data, eval_labels = _load_testing_data(args.train)

    mnist_classifier = model(train_data, train_labels, eval_data, eval_labels)

    if args.current_host == args.hosts[0]:
        # save model to an S3 directory with version number '00000001' in Tensorflow SavedModel Format
        # To export the model as h5 format use model.save('my_model.h5')
        mnist_classifier.export(os.path.join(args.sm_model_dir, '000000001'))


# Create a training job using ModelTrainer

`ModelTrainer` orchestrates training jobs on SageMaker infrastructure. Key components:

- **`training_image`**: Docker image URI containing TensorFlow (retrieved via `image_uris.retrieve()`)
- **`SourceCode`**: Specifies your training script (`entry_script`) and its directory (`source_dir`)
- **`Compute`**: Defines instance type and count for training
- **`hyperparameters`**: Dictionary of training parameters (values must be strings)

**Distributed Training:** This example trains on 2 instances using TensorFlow's parameter server strategy. The training script handles distributed coordination using the `SM_HOSTS` environment variable.

In [ ]:
# cell 04
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.core import image_uris

image_uri = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.16",
    py_version="py310",
    instance_type="ml.c5.xlarge",
    image_scope="training",
)

mnist_estimator = ModelTrainer(
    training_image=image_uri,
    role=role,
    source_code=SourceCode(source_dir="mnist-tensorflow", entry_script="train.py"),
    compute=Compute(instance_type="ml.c5.xlarge", instance_count=1),
    sagemaker_session=sagemaker_session,
)

# Calling `train`

Call `trainer.train()` to start the training job. The `InputData` dataclass maps a channel name to an S3 data source:

- **`channel_name`**: Name of the input channel (accessible via `SM_CHANNEL_<NAME>` in the script)
- **`data_source`**: S3 URI containing the training data

SageMaker downloads the data to the container and makes it available at the channel path. When training completes, the saved model is uploaded to S3.

Training with TensorFlow 2.x:

In [ ]:
# cell 05
from sagemaker.core.training.configs import InputData

mnist_estimator.train(input_data_config=[InputData(channel_name="training", data_source=training_data_uri)])

# Deploy the trained model to an endpoint

`ModelBuilder` packages your model and deploys it as a real-time inference endpoint.

For TensorFlow models saved in SavedModel format, the TensorFlow Serving container handles inference automatically - no custom inference code is needed. The container:

- Loads the SavedModel from the model artifacts
- Exposes a REST API compatible with TensorFlow Serving
- Handles input/output serialization for numpy arrays and JSON

Key `ModelBuilder` parameters:
- **`model`**: The trainer object (to get model artifacts location)
- **`schema_builder`**: Defines input/output shapes for serialization
- **`role_arn`**: IAM role for the endpoint

In [ ]:
inference_image_uri = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.16",
    py_version="py310",
    instance_type="ml.c5.xlarge",
    image_scope="inference",
)

In [ ]:
# cell 06
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
import numpy as np

schema_builder = SchemaBuilder(
    sample_input=np.zeros((1, 28, 28), dtype=np.float32),  # MNIST input shape
    sample_output=np.zeros((1, 10), dtype=np.float32),     # 10 classes output
)

model_data_url = mnist_estimator._latest_training_job.model_artifacts.s3_model_artifacts

model_builder = ModelBuilder(
    image_uri=inference_image_uri,
    s3_model_data_url=model_data_url,
    model=mnist_estimator,
    schema_builder=schema_builder,
    role_arn=role,
)

model_builder.build()

In [ ]:
endpoint = model_builder.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge")

# Invoke the endpoint

Let's download the training data and use that as input for inference.

In [ ]:
# cell 07
import numpy as np

!aws --region {region} s3 cp s3://sagemaker-sample-data-{region}/tensorflow/mnist/train_data.npy train_data.npy
!aws --region {region} s3 cp s3://sagemaker-sample-data-{region}/tensorflow/mnist/train_labels.npy train_labels.npy

train_data = np.load('train_data.npy')
train_labels = np.load('train_labels.npy')

Send inference requests using `endpoint.invoke()`:

- **`body`**: Serialized input data (JSON string)
- **`content_type`**: MIME type of the request body

The TensorFlow Serving container accepts numpy arrays serialized as JSON and returns predictions in the same format.

In [ ]:
# cell 08
import json

response = endpoint.invoke(
    body=json.dumps(train_data[:50].tolist()),
    content_type="application/json"
)

result = json.loads(response.body.read().decode("utf-8"))
predictions = result.get('predictions', result)  # Handle both TF Serving formats

for i in range(0, 50):
    prediction = np.argmax(predictions[i])
    label = train_labels[i]
    print('prediction is {}, label is {}, matched: {}'.format(prediction, label, prediction == label))

# Delete the endpoint

Delete the endpoint when finished to stop incurring charges.

In [ ]:
# cell 09
sagemaker_session.delete_endpoint(endpoint_name=endpoint.endpoint_name)